In [4]:
from lczero.backends import Weights, Backend, GameState
w = Weights('models/weights-256_10.pb')


In [ ]:
#print(w.filters())

: 

In [ ]:
Backend.available_backends()

['eigen',
 'trivial',
 'random',
 'check',
 'recordreplay',
 'roundrobin',
 'multiplexing',
 'demux']

In [5]:
b = Backend(weights=w)

Creating backend [eigen]...
Using Eigen version 3.4.0
Eigen max batch size is 256.


In [ ]:
args = [el for el in dir(b) if not el.startswith('__')]
print(args)

['available_backends', 'capabilities', 'evaluate']


In [3]:
g = GameState(moves=['e2e4', 'e7e5'])
print(g.as_string())

https://lc0.org/fen/rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR_w_KQkq_-


In [8]:
def dir_c(a):
    return [item for item in dir(a) if not item.startswith('__')]

In [7]:
print(dir_c(b))

['available_backends', 'capabilities', 'evaluate']


In [ ]:
# 1. Создание первого входа из объекта g (предполагается, что g и b уже определены ранее)
i1 = g.as_input(b)

# 2. Создание второго входа из новой позиции FEN (исправлено: добавлен аргумент backend 'b')
i2 = GameState(fen='2R5/5kpp/4p3/p4p2/3B4/1K5N/4rNPP/8 b - - 0 29').as_input(b)

# 3. Проверка маски для первого состояния (вывод в консоль)
print(bin(i1.mask(0)))

# 4. Проверка маски для второго состояния (вывод в консоль)
print(bin(i2.mask(0)))

# 5. Оценка обоих состояний движком
o1, o2 = b.evaluate(i1, i2)

# 6. Вывод оценки качества (Q-value) для первой позиции
print(o1.q())

# 7. Вывод оценки качества (Q-value) для второй позиции
print(o2.q())

# 8. Получение списка ходов и вероятностей политики (softmax) для первой позиции
# Исправлено: использован объект 'g' вместо несуществующего 'p1'
policy_data = list(zip(g.moves(), o1.p_softmax(*g.policy_indices())))

# 9. Вывод результатов (ход -> вероятность)
for move, prob in policy_data:
    print(f"{move}: {prob}")

In [ ]:
print(dir_c(i2))
print()

['clone', 'mask', 'set_mask', 'set_val', 'val']
None


In [ ]:
!unzip tf_model.zip

Archive:  tf_model.zip
   creating: tf_model/assets/
  inflating: tf_model/keras_metadata.pb  
  inflating: tf_model/saved_model.pb  
   creating: tf_model/variables/
  inflating: tf_model/variables/variables.data-00000-of-00001  
  inflating: tf_model/variables/variables.index  


In [ ]:
import tensorflow as tf


2026-02-25 10:11:03.922268: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-25 10:11:04.804309: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-25 10:11:08.242243: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
try:
    model = tf.saved_model.load('tf_model')
except Exception as e:
    print(f"Exception occured: \n\n {e}")

Instructions for updating:
Use `tf.saved_model.load` instead.
Exception occured: 

 load() missing 2 required positional arguments: 'tags' and 'export_dir'


In [ ]:
!unzip tf_model.zip 

Archive:  tf_model.zip
   creating: tf_model/assets/
  inflating: tf_model/keras_metadata.pb  
  inflating: tf_model/saved_model.pb  
   creating: tf_model/variables/
  inflating: tf_model/variables/variables.data-00000-of-00001  
  inflating: tf_model/variables/variables.index  


In [19]:
!gzip -d training-run1--20200711-2017/training.96193352.gz

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model('models/tf_model_19x256.keras')
model.summary()

In [ ]:
inputs = model.layers[0].input
new_model = tf.keras.Model(inputs=model.input, 
                          outputs=model.layers[-3].output)
new_model.summary()

In [ ]:
pred = new_model(np.random.randn(5,8,8,112))
print(np.mean(pred), np.std(pred))

13272.448 25374.87


In [ ]:
import tensorflow as tf
@tf.function
def mse(y_true, y_pred):
    print(y_pred.shape)
    return tf.sum((y_true-y_pred)**2)

2026-04-03 12:30:34.866371: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [7]:
import numpy as np
y_true = tf.constant([1,2], dtype=tf.float16)
y_pred = tf.Variable(np.array([1, 1]), dtype=tf.float16)
loss = mse(y_true, y_pred)
loss.backward()

NotImplementedError: in user code:

    File "/tmp/ipykernel_59697/2284551555.py", line 4, in mse  *
        return np.sum((y_true-y_pred)**2)
    File "/home/demid/Code/Python/ChessErrorClassification/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py", line 2466, in sum  **
        return _wrapreduction(
    File "/home/demid/Code/Python/ChessErrorClassification/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py", line 86, in _wrapreduction
        return ufunc.reduce(obj, axis, dtype, out, **passkwargs)

    NotImplementedError: Cannot convert a symbolic tf.Tensor (pow:0) to a numpy array. This error may indicate that you're trying to pass a Tensor to a NumPy call, which is not supported.
